# 03 - Expression & Spatial Metadata, Dissected

## Learning objectives
1. Inspect `adata.X`, `.obs`, `.var`, `.obsm['spatial']`, `.uns['spatial']`.
2. Read off the number of spots and genes and peek at metadata.
3. Understand the spatial-coordinate structure and the image scale factors.
4. Make image-to-expression **registration** concrete with one multiply.

## Concept
We revisit the 'four data objects' from notebook 00, now on the real dataset. The goal is
that you can always answer: *where is the matrix? where are the coordinates? what units
are they in? how do they map onto the image?*


In [ ]:
# --- Standard setup: make `utils` importable and seed RNGs ---
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'utils').exists():
    ROOT = ROOT.parent  # in case the notebook is opened from a subfolder
sys.path.insert(0, str(ROOT))

from utils import st_helpers as st
st.set_seeds()  # reproducibility (seed = 0)
print('Project root:', st.project_root())


In [ ]:
adata = st.load_adata('adata_raw.h5ad')
adata


### Spots and genes (matrix shape)

In [ ]:
import numpy as np

n_spots, n_genes = adata.shape
print(f'Number of spots (obs): {n_spots:,}')
print(f'Number of genes (var): {n_genes:,}')
print('Type of .X         :', type(adata.X))
print('.X dtype           :', adata.X.dtype)


`.X` is typically a **sparse** matrix (most entries are 0 - a given spot expresses only a
fraction of all genes). This is like a mostly-empty volume; storing only non-zeros saves
huge memory.

In [ ]:
# Look at a tiny dense corner of the count matrix.
import scipy.sparse as sp

X = adata.X
corner = X[:5, :8].toarray() if sp.issparse(X) else np.asarray(X[:5, :8])
print('First 5 spots x 8 genes (UMI counts):')
print(corner)

density = (X.nnz / (n_spots * n_genes)) if sp.issparse(X) else np.mean(X != 0)
print(f'\nMatrix density (fraction non-zero): {density:.3f}')


**Expected output:** small integer UMI counts (many zeros) and a density well below 0.2 -
i.e. the matrix is sparse.

### `.obs` - per-spot metadata (rows of the matrix)

In [ ]:
print('obs columns:', list(adata.obs.columns))
adata.obs.head()


Each row is one spot. Columns may include in-tissue flags and array row/col indices.
(Quality-control columns like `total_counts` get added in notebook 04.)

### `.var` - per-gene metadata (columns of the matrix)

In [ ]:
print('var columns:', list(adata.var.columns))
print('first 10 gene names:', list(adata.var_names[:10]))
adata.var.head()


Note the gene-name casing (mouse: `Snap25`; human would be `SNAP25`). This is why we
always filter marker lists through `st.genes_present()`.

### `.obsm['spatial']` - spot coordinates

In [ ]:
coords = adata.obsm['spatial']
print('coords shape:', coords.shape, '(n_spots, 2) -> columns are [x_pixel, y_pixel]')
print('x range:', coords[:, 0].min(), '..', coords[:, 0].max())
print('y range:', coords[:, 1].min(), '..', coords[:, 1].max())
print('first 5 coords:\n', coords[:5])


These are **full-resolution image pixels**. They are the link between a matrix row and a
location on the H&E photograph.

### `.uns['spatial']` - the image and scale factors

In [ ]:
lib = st.get_library_id(adata)
sf = st.get_scalefactors(adata)
for k, v in sf.items():
    print(f'{k:>28}: {v}')

hires = st.get_image(adata, 'hires')
lowres = st.get_image(adata, 'lowres')
print('\nhires image array shape :', hires.shape)
print('lowres image array shape:', lowres.shape)


Key scale factors:
- `tissue_hires_scalef` / `tissue_lowres_scalef` - multiply a full-res pixel coordinate by
  this to land on the `hires` / `lowres` image.
- `spot_diameter_fullres` - the spot's diameter **in full-resolution pixels** (we use this
  in notebook 09 to size image patches).

### Registration in one multiply
Let's overlay the spots on the hires image to *prove* the coordinates and scale factor are
consistent. (Notebook 06 does the polished version; this is the bare-bones check.)

In [ ]:
import matplotlib.pyplot as plt

scalef = sf['tissue_hires_scalef']
xy_hires = coords * scalef  # full-res pixels -> hires-image pixels

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(hires)
ax.scatter(xy_hires[:, 0], xy_hires[:, 1], s=4, c='cyan', alpha=0.5)
ax.set_title('Spots (cyan) overlaid on hires H&E via scale-factor multiply')
ax.axis('off')
plt.show()


**Expected output:** cyan dots tiling exactly over the tissue in the H&E image. If the
dots are shifted or shrunk into a corner, you forgot the scale-factor multiply (a classic
registration bug).

## The six distinctions, on real data
| Concept | This dataset |
|---|---|
| Tissue image pixels | `hires`/`lowres` RGB arrays in `uns['spatial']` |
| Visium spots | the ~2,700 rows of `adata` |
| Count matrix | `adata.X` (sparse UMI counts) |
| Spatial coordinates | `adata.obsm['spatial']` (full-res pixels) |
| Scale factors | `uns['spatial'][lib]['scalefactors']` |
| Registration | coordinate x `tissue_hires_scalef` |

## Common pitfalls
- Plotting raw coordinates on the (downscaled) hires image without the scale factor.
- Assuming `.X` is dense (it is usually sparse - convert small slices only).
- Confusing array indices (row/col on the spot grid) with pixel coordinates.

## Interpretation
You can now navigate any Visium AnnData blindfolded and explain how the matrix registers
to the image.

## What this means biologically
Each matrix row is a real, locatable patch of brain tissue. Keeping the row<->pixel link
intact is what lets us ask spatial biological questions later.

---
**Next:** `04_qc_and_preprocessing.ipynb` - clean and normalize the data.
